In [422]:
import argparse
import random
import torch
import numpy as np

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color, draw_geometries
from geotransformer.utils.registration import compute_registration_error

from config import make_cfg
from model import create_model

import open3d as o3d

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

----------------

# Customizable params to see what breaks

In [ ]:
scale = 1.0
ratio = 1.0

------------------

In [424]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [425]:
WEIGHTS = "../../output/with_aug/snapshots/epoch-40.pth.tar"
cfg = make_cfg()
model = create_model(cfg).cuda()
state_dict = torch.load(WEIGHTS)
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [426]:
#REF_NUM = 20
REF_NUM = 25

SRC_FILE = f"../../data/faces/demo/src_{REF_NUM}.npy"
REF_FILE = f"../../data/faces/demo/ref_{REF_NUM}.npy"
GT_FILE = f"../../data/faces/demo/gt_{REF_NUM}.npy"
MORPHED_FULL_FILE = f"../../data/faces/demo/morphed_full_{REF_NUM}.npy"

In [427]:
def load_data(scale=1.0, ratio = 1.0):
    src_points = np.load(SRC_FILE)

    # scaling
    src_points *= scale

    # nonuniform downsampling
    num_points = src_points.shape[0]
    indices = np.random.choice(num_points, int(ratio*num_points), replace=False)
    src_points = src_points[indices]
    

    ref_points = np.load(REF_FILE)
    morphed_full_points = np.load(MORPHED_FULL_FILE)
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
        "morphed_full": morphed_full_points.astype(np.float32),
        "gt_z": np.zeros((32, 100), dtype=np.float32) 
    }

    if GT_FILE is not None:
        transform = np.load(GT_FILE)
        data_dict["transform"] = transform.astype(np.float32)

    return data_dict

def open3d_webrtc_draw(geometries):
    o3d.visualization.draw(geometries)
   

In [428]:
data_dict = load_data(scale=scale, ratio=ratio)
data_dict.keys()

dict_keys(['ref_points', 'src_points', 'ref_feats', 'src_feats', 'morphed_full', 'gt_z', 'transform'])

### Run Model w/ ref and src

In [429]:
# prepare data
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)
# get results
ref_length_orig = data_dict['lengths'][0][0].item()
orig_points = data_dict['points'][0] # [ref; src]
ref_points = orig_points[:ref_length_orig]
src_points = orig_points[ref_length_orig:]

# prediction
data_dict = to_cuda(data_dict)
output_dict = model(data_dict)
data_dict = release_cuda(data_dict)
output_dict = release_cuda(output_dict)

estimated_transform = output_dict["estimated_transform"]
transform = data_dict["transform"]

### Visualize ref and initial src

In [430]:
# visualization
ref_pcd = make_open3d_point_cloud(ref_points)
ref_pcd.estimate_normals()
ref_pcd.paint_uniform_color(get_color("custom_blue"))
src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color(get_color("custom_yellow"))
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

### Visualize ref and src w/ estimated transform

In [431]:
estimated_t_src_pcd = src_pcd.transform(estimated_transform)
o3d.visualization.draw_plotly([ref_pcd, estimated_t_src_pcd])

# compute error
rre, rte = compute_registration_error(transform, estimated_transform)
print(f"RRE(deg): {rre:.3f}, RTE(m): {rte:.3f}")

RRE(deg): 0.000, RTE(m): 0.029


### Visualize ref and src w/ gt transform

In [432]:
# original code that forgot to add scaling to gt transform

# new_src_pcd = make_open3d_point_cloud(src_points)
# new_src_pcd.transform(transform)
# o3d.visualization.draw_plotly([ref_pcd, new_src_pcd])

In [433]:
# The factor needed to bring scale (e.g., 1.2) down to 1.0
inverse_scale = 1.0 / scale

# Prepare the transformation matrix
new_transform = transform.copy()

# Scale down the rotation/basis component
new_transform[:3, :3] *= inverse_scale

# Apply to the point cloud
new_src_pcd = make_open3d_point_cloud(src_points)
new_src_pcd.transform(new_transform)

o3d.visualization.draw_plotly([ref_pcd, new_src_pcd])

### Visualize src w/ estimated transform vs. gt transform 

In [434]:
o3d.visualization.draw_plotly([estimated_t_src_pcd, new_src_pcd])

### Visualize morphed ref w/ predicted coeffs vs. gt coeffs

In [435]:
pred_morphed_data = output_dict["morphed_full"]

if torch.is_tensor(pred_morphed_data):
    pred_morphed_data = pred_morphed_data.detach().cpu().numpy()
if pred_morphed_data.ndim == 3:
    pred_morphed_data = pred_morphed_data.squeeze(0)

pred_morphed_pcd = o3d.geometry.PointCloud()
pred_morphed_pcd.points = o3d.utility.Vector3dVector(pred_morphed_data)
pred_morphed_pcd.estimate_normals()
pred_morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green = Predicted Model Output

recon_gt_data = output_dict["recon_gt_points"]
if torch.is_tensor(recon_gt_data):
    recon_gt_data = recon_gt_data.detach().cpu().numpy()
if recon_gt_data.ndim == 3:
    recon_gt_data = recon_gt_data.squeeze(0)

recon_gt_pcd = o3d.geometry.PointCloud()
recon_gt_pcd.points = o3d.utility.Vector3dVector(recon_gt_data)
recon_gt_pcd.estimate_normals()
recon_gt_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue = GT PCA Reconstruction

print("Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)")
o3d.visualization.draw_plotly([pred_morphed_pcd, recon_gt_pcd])


Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)


### Visualize final reconstruction and alignment
- Green --> ref morphed by predicted coeffs
- Blue --> src aligned by estimated transform

In [436]:
estimated_t_src_pcd.paint_uniform_color([0, 0, 0.5])
o3d.visualization.draw_plotly([pred_morphed_pcd, estimated_t_src_pcd])

------------------